In [7]:
# Save the full JSON output for each Italian recipe
import os

output_dir = "italian_recipes_json"
os.makedirs(output_dir, exist_ok=True)

for meal_name in meal_names:
    recipe_name = format_name(meal_name)
    url = f"https://www.themealdb.com/api/json/v1/1/search.php?s={meal_name}"
    resp = requests.get(url)
    meal_data = resp.json()
    if not meal_data or not meal_data.get('meals'):
        continue
    file_path = os.path.join(output_dir, f"{recipe_name}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(meal_data, f, ensure_ascii=False, indent=2)
print(f"Saved full JSON for {len(meal_names)} recipes to {output_dir}/")

Saved full JSON for 21 recipes to italian_recipes_json/


In [6]:
# Fetch all Italian recipes and save each as a separate JSON file
import os

output_dir = "italian_recipes_json"
os.makedirs(output_dir, exist_ok=True)

for meal in detailed_meals:
    recipe_name = format_name(meal['recipe'])
    recipe_data = {
        "name": meal['recipe'],
        "ingredients": meal['ingredients']
    }
    file_path = os.path.join(output_dir, f"{recipe_name}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(recipe_data, f, ensure_ascii=False, indent=2)
print(f"Saved {len(detailed_meals)} recipes to {output_dir}/")

Saved 18 recipes to italian_recipes_json/


# Extract Italian Meal Ingredients from TheMealDB

This notebook fetches all Italian meals from TheMealDB API, retrieves their ingredients, and formats them as `Recipe` and `Uses` predicates as specified.

In [1]:
# Import Required Libraries
import requests
import json


## Fetch List of Italian Meals
Use the API to get all Italian meals and extract their names.

In [2]:
# Fetch the list of Italian meals
italian_meals_url = "https://www.themealdb.com/api/json/v1/1/filter.php?a=Italian"
response = requests.get(italian_meals_url)
data = response.json()
meals = data.get('meals', [])
meal_names = [meal['strMeal'] for meal in meals]
print(f"Found {len(meal_names)} Italian meals.")
meal_names[:5]  # Show a sample

Found 21 Italian meals.


['Budino Di Ricotta',
 'Chicken Alfredo Primavera',
 'Chilli prawn linguine',
 'Fettuccine Alfredo',
 'Fettucine alfredo']

## Retrieve Ingredients for Each Meal
For each meal, fetch its details and extract the list of ingredients. Skip the specified recipes.

In [3]:
# Helper functions for formatting
import re

def format_name(name):
    return re.sub(r'\s+', '_', name.strip().lower())

skip_recipes = {'vegan_lasagna', 'spaghetti_bolognese', 'spicy_arrabiata_penne'}

# Retrieve ingredients for each meal
detailed_meals = []
for meal_name in meal_names:
    recipe_name = format_name(meal_name)
    if recipe_name in skip_recipes:
        continue
    url = f"https://www.themealdb.com/api/json/v1/1/search.php?s={meal_name}"
    resp = requests.get(url)
    meal_data = resp.json().get('meals')
    if not meal_data:
        continue
    meal = meal_data[0]
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f'strIngredient{i}')
        if ing and ing.strip():
            ingredients.append(ing.strip())
    detailed_meals.append({'recipe': meal_name, 'ingredients': ingredients})

# Show a sample
print(detailed_meals[0])

{'recipe': 'Budino Di Ricotta', 'ingredients': ['Ricotta', 'Eggs', 'Flour', 'Sugar', 'Cinnamon', 'Lemons', 'Dark Rum', 'Icing Sugar']}


## Format Recipe and Uses Predicates
Format the meal and ingredient names as required for the predicates.

In [4]:
# Format predicates
predicates = []
for meal in detailed_meals:
    recipe_name = format_name(meal['recipe'])
    predicates.append(f"Recipe({recipe_name}).")
    for ing in meal['ingredients']:
        ing_name = format_name(ing)
        predicates.append(f"Uses({recipe_name}, {ing_name}).")

## Output Recipe and Uses Predicates
Display all generated predicates.

In [5]:
# Output all predicates
for p in predicates:
    print(p)

Recipe(budino_di_ricotta).
Uses(budino_di_ricotta, ricotta).
Uses(budino_di_ricotta, eggs).
Uses(budino_di_ricotta, flour).
Uses(budino_di_ricotta, sugar).
Uses(budino_di_ricotta, cinnamon).
Uses(budino_di_ricotta, lemons).
Uses(budino_di_ricotta, dark_rum).
Uses(budino_di_ricotta, icing_sugar).
Recipe(chicken_alfredo_primavera).
Uses(chicken_alfredo_primavera, butter).
Uses(chicken_alfredo_primavera, olive_oil).
Uses(chicken_alfredo_primavera, chicken).
Uses(chicken_alfredo_primavera, salt).
Uses(chicken_alfredo_primavera, squash).
Uses(chicken_alfredo_primavera, broccoli).
Uses(chicken_alfredo_primavera, mushrooms).
Uses(chicken_alfredo_primavera, pepper).
Uses(chicken_alfredo_primavera, onion).
Uses(chicken_alfredo_primavera, garlic).
Uses(chicken_alfredo_primavera, red_pepper_flakes).
Uses(chicken_alfredo_primavera, white_wine).
Uses(chicken_alfredo_primavera, milk).
Uses(chicken_alfredo_primavera, heavy_cream).
Uses(chicken_alfredo_primavera, parmesan_cheese).
Uses(chicken_alfredo